#### Realistic Romina PLC EUDR Dataset Creation

Focus Areas (Based on Romina PLC actual operations):
- Sidama Region: Chiri, Bensa, Bona Zuriya, Aroresa, Wensho
- Yirgacheffe (Gedeo Zone): Yirgacheffe, Kochere, Gedeb, Wenago, Bule

Coffee Type: Washed & Natural Arabica  
Characteristics: Floral, citrus, bright acidity, jasmine, tea-like

This dataset is designed for EUDR compliance (plot-level GPS, deforestation-free proof, legality, traceability)

#### Setup

In [1]:
import os
import pandas as pd
import numpy as np
from datetime import datetime

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

folders = [
    'data/raw', 
    'data/processed', 
    'outputs/viz', 
    'app'
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project structure initialized.")
print(f"Current Working Directory: {os.getcwd()}")

Project structure initialized.
Current Working Directory: C:\Users\tohiba\Desktop\eudr-coffee-traceability-romina


#### Geographic & Administrative Configuration

In [6]:
np.random.seed(42)

regions_map = {
    'Sidama': ['Chire', 'Bensa', 'Bona Zuria', 'Aroresa', 'Wensho'],
    'South Ethiopia': ['Yirgacheffe', 'Kochere', 'Gedeb', 'Wenago', 'Bule']
}

centers = {
    'Chire': (6.56, 38.74), 'Bensa': (6.52, 38.79), 'Bona Zuria': (6.48, 38.48),
    'Aroresa': (6.35, 38.98), 'Wensho': (6.66, 38.52),
    'Yirgacheffe': (6.16, 38.20), 'Kochere': (6.02, 38.22), 'Gedeb': (5.92, 38.25),
    'Wenago': (6.33, 38.26), 'Bule': (6.30, 38.40)
}

#### Core Dataset Generation Engine

In [8]:
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

n_plots = 2000
regions = np.random.choice(['Sidama', 'South Ethiopia'], n_plots, p=[0.55, 0.45])
woredas = [np.random.choice(regions_map[r]) for r in regions]

data = {
    'farmer_id': [f'ROM-F-{i:05d}' for i in range(1, n_plots + 1)],
    'plot_id': [f'ROM-P-{i:05d}' for i in range(1, n_plots + 1)],
    'woreda': woredas,
    'region': regions,
    'latitude': np.round([centers[w][0] + np.random.uniform(-0.05, 0.05) for w in woredas], 6),
    'longitude': np.round([centers[w][1] + np.random.uniform(-0.05, 0.05) for w in woredas], 6),
    
    # Altitude: High-altitude specialty belt (1800m - 2200m)
    'altitude_masl': np.random.randint(1850, 2250, n_plots),
    
    # Plot size: Typical Ethiopian smallholder (mostly < 2ha)
    'plot_size_ha': np.random.lognormal(mean=-0.2, sigma=0.5, size=n_plots).round(3),
    
    # EUDR Requirement: Evidence of no deforestation after Dec 31, 2020
    'last_deforestation_event': np.random.choice(
        [pd.Timestamp('2010-01-01'), pd.Timestamp('2015-06-15'), pd.Timestamp('2021-03-10')], 
        n_plots, p=[0.70, 0.25, 0.05]
    ),
    
    'legality_status': np.random.choice(['Verified', 'Pending', 'Incomplete'], n_plots, p=[0.85, 0.10, 0.05]),
    'coffee_variety': '74110 / 74112 (Specialty Arabica)', # Common resilient varieties in Ethiopia
}

df = pd.DataFrame(data)

# Calculate Harvest Dates (2024/25 Season)
# Main harvest: Oct to Jan
start_dates = [pd.Timestamp('2024-10-15') + timedelta(days=np.random.randint(0, 45)) for _ in range(n_plots)]
df['harvest_start'] = start_dates
df['harvest_end'] = [d + timedelta(days=np.random.randint(30, 60)) for d in start_dates]

print("Agricultural and EUDR base data generated.")

Agricultural and EUDR base data generated.


#### The Proprietary EUDR Scoring Engine

In [9]:
def calculate_eudr_compliance(row):
    score = 40
    
    # 1. Deforestation Check (The most critical EUDR pillar)
    # If deforestation occurred after 2020-12-31, the score drops significantly
    cutoff_date = pd.Timestamp('2020-12-31')
    if row['last_deforestation_event'] > cutoff_date:
        score -= 40 # Automatic non-compliance risk
    else:
        score += 30
        
    # 2. Legality Check (Land use rights)
    if row['legality_status'] == 'Verified':
        score += 20
    elif row['legality_status'] == 'Pending':
        score += 10
        
    # 3. Precision Check (Placeholder for polygon vs point data)
    # EUDR requires polygons for plots > 4 hectares
    if row['plot_size_ha'] > 4.0:
        score -= 5 # Higher complexity/risk for larger plots without polygons
    else:
        score += 10
        
    return np.clip(score + np.random.normal(0, 2), 0, 100)

df['eudr_readiness_score'] = df.apply(calculate_eudr_compliance, axis=1).round(1)

print("EUDR Compliance scores calculated based on 2020 cutoff.")

EUDR Compliance scores calculated based on 2020 cutoff.


#### Data Export & Verification

In [10]:
output_path = 'data/raw/romina_coffee_traceability_master.csv'
df.to_csv(output_path, index=False)

# Quick Analysis of "EUDR Failures"
failed_plots = len(df[df['last_deforestation_event'] > pd.Timestamp('2020-12-31')])

print(f"Master Dataset Saved: {output_path}")
print("-" * 40)
print(f"Season: 2024/25")
print(f"EUDR Non-Compliant Plots (Post-2020 Deforestation): {failed_plots}")
print(f"Average System Readiness: {df['eudr_readiness_score'].mean():.1f}%")
print("-" * 40)
df.head()

Master Dataset Saved: data/raw/romina_coffee_traceability_master.csv
----------------------------------------
Season: 2024/25
EUDR Non-Compliant Plots (Post-2020 Deforestation): 85
Average System Readiness: 94.4%
----------------------------------------


,farmer_id,plot_id,woreda,region,latitude,longitude,altitude_masl,plot_size_ha,last_deforestation_event,legality_status,coffee_variety,harvest_start,harvest_end,eudr_readiness_score
0,ROM-F-00001,ROM-P-00001,Chire,Sidama,6.560150,38.786712,2093,0.776,2010-01-01,Verified,74110 / 74112 (Specialty Arabica),2024-11-18,2025-01-16,100.0
1,ROM-F-00002,ROM-P-00002,Bona Zuria,Sidama,6.454769,38.499211,2162,1.016,2010-01-01,Pending,74110 / 74112 (Specialty Arabica),2024-11-05,2024-12-22,91.8
2,ROM-F-00003,ROM-P-00003,Wenago,South Ethiopia,6.348516,38.235132,1925,1.651,2010-01-01,Verified,74110 / 74112 (Specialty Arabica),2024-11-15,2025-01-12,100.0
3,ROM-F-00004,ROM-P-00004,Chire,Sidama,6.580426,38.780968,2218,1.529,2015-06-15,Verified,74110 / 74112 (Specialty Arabica),2024-11-09,2024-12-21,100.0
4,ROM-F-00005,ROM-P-00005,Bona Zuria,Sidama,6.465797,38.459403,2111,0.883,2010-01-01,Verified,74110 / 74112 (Specialty Arabica),2024-10-20,2024-12-08,97.7
